# Claude API Setup

In [ ]:
#  Install Dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load variables from .env file (keeps secret key in .env since .env is not pushed into vrs control tools)
from dotenv import load_dotenv
load_dotenv()

# Create Anthropic Client
from anthropic import Anthropic
client = Anthropic()
model="claude-sonnet-4-0" # can be any model

In [ ]:
#  Making Requests (stateless)
message = client.messages.create(
    model=model,
    max_tokens=1000, # Token limit per message, can be changed
    messages=[
        {
            "role": "user",
            "content":"What is Quantum Computing?" # Actual prompt
        }
    ]
)

# To see generated answer without other metadata
message.content[0].text

# Chatbot Implementation (stateful)

In [ ]:
# =============================================================================================================
# Stateful chatbot implementation
# =============================================================================================================

# Because Claude API are stateless, you need to maintain a list of historical messages to make it stateful

# Helpers to be used throughout
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
    model=model,
    max_tokens=1000, # Token limit per message, can be changed
    messages=messages, # Stateful
    )
    return message.content[0].text

# implementation

# 1. init starting list
messages = []

# 2. Add the initial user query and append it to messages[]
add_user_message(messages, "What is Quantum Computing?")

# 3. Get answer
answer = chat(messages)
answer

# 4. Take answer and add the generated message (assistant) into message[]
add_assistant_message(messages, answer)

# Repeat steps 2-4 for every chain of question-answer message

In [ ]:
# =============================================================================================================
# Stateful chatbot implementation Pt2. (with while loop)
# =============================================================================================================

messages = []

while True:
    # Get user input with VSC's built-in input function
    user_input = input("> ")
    print(">", user_input)

    add_user_message(messages, user_input)
    answer = chat(messages)
    add_assistant_message(messages, answer)

    print(answer)

# Adding system prompts

In [ ]:
# =============================================================================================================
# System prompting - add system prompt to def chat(message) function
# =============================================================================================================
def chat(messages, system=None):
    # params in dict allows system prompt to be None if it isn't added
    params = {
        "model": model,
        "max_tokens": 1000, # Token limit per message, can be changed
        "messages": messages, # Stateful
    }

    # If sysytem prompt was passed in, add a system key into the params dict
    if system:
        params["system"] = system

    message = client.messages.create(**params) # '**' unpacks params dict to say model=model, max_tokens=1000, messages=messages (because thats the format for the API)
    return message.content[0].text

messages = []
system = """
blablahblah
"""
add_user_message(messages, "something the user asks")
answer = chat(messages, system=system) # Works with system prompt
answer = chat(messages) # Also works withous system prompt

# Streaming to allow line by line output

In [ ]:
messages = []
add_user_message(messages, "something the user asks")
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

"""
What we will see is a chunk of:
1) MessageStart - signals start
2) ContentBlockStart - signals start of new block, tool-use, etc.
3) ContentBlockDelta - actual generated text output (usually alot of these blocks)
4) ContentBlockStop - signals end of current block
5) MessageDelta - signals message is complete
6) MessageStop - signals end
"""

# ==== To make the streaming easier to use ====

messages = []
add_user_message(messages, "something the user asks")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:                # Difference
    for text in stream.text_stream:
        print(text, end="") # end arg just make sures that text a streamed side by side instead of in new line

# Uncomment to get final message of streamed output (for storing in database)
# stream.get_final_message()

# Structure JSON outputs with no header/footer commentaries

In [ ]:
messages = []
add_user_message(messages, "something the user asks")
add_assistant_message(messages, "```json") # Tells claude you want everything after the ```json output
text = chat(messages, stop_sequences=["```"]) # Tells claude you want everything before the closing ``` sequence
text

import json
json.loads(text.strip()) # remove \n using strip

"""
Additional learning pointer:
instead of just using add_assistant_message(messages, "```json")
you can use add_assistant_message(messages, "```bash") or even add a message before the ```json to say you dont want ay additonal comments e.g.
add_assistant_message(messages, Dont add any Here are the comments with no additional comments:\n"```json")
"""